In [4]:
import pandas as pd

In [5]:
prod = pd.read_csv("../data/raw/production_consumption_DK2_2024_01_01_2025_02_20.csv")

prod.head()

,HourUTC,HourDK,PriceArea,CentralPowerMWh,LocalPowerMWh,CommercialPowerMWh,LocalPowerSelfConMWh,OffshoreWindLt100MW_MWh,OffshoreWindGe100MW_MWh,OnshoreWindLt50kW_MWh,...,ExchangeSE_MWh,ExchangeGE_MWh,ExchangeNL_MWh,ExchangeGB_MWh,ExchangeGreatBelt_MWh,GrossConsumptionMWh,GridLossTransmissionMWh,GridLossInterconnectorsMWh,GridLossDistributionMWh,PowerToHeatMWh
0,2023-12-31T23:00:00,2024-01-01T00:00:00,DK2,299.433470,40.874795,199.417036,15.612938,22.6388,468.954368,0.264673,...,-1226.629258,966.118984,NaN,NaN,530.7,1514.103305,71.590920,11.202016,60.617714,67.777938
1,2024-01-01T00:00:00,2024-01-01T01:00:00,DK2,294.905852,40.678208,200.459208,13.151875,19.0601,389.895969,0.202649,...,-1166.471250,986.252008,NaN,NaN,532.6,1489.812135,59.079085,11.256992,59.830525,72.936872
2,2024-01-01T01:00:00,2024-01-01T02:00:00,DK2,293.800519,41.743095,198.599972,13.632495,12.2958,510.685968,0.242571,...,-1311.097968,986.575000,NaN,NaN,531.6,1464.270618,64.467446,11.192000,59.075538,71.830910
3,2024-01-01T02:00:00,2024-01-01T03:00:00,DK2,275.398603,46.746541,201.408134,14.913982,15.6522,517.489167,0.269561,...,-1365.566234,986.662992,NaN,NaN,529.2,1421.999512,64.123213,11.403008,57.865027,69.777540
4,2024-01-01T03:00:00,2024-01-01T04:00:00,DK2,275.347865,44.680464,202.559328,14.601559,14.7248,617.276566,0.302115,...,-1472.738500,986.767000,NaN,NaN,490.7,1389.341870,70.226677,11.290000,56.259960,65.599544


In [6]:
prod.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9984 entries, 0 to 9983
Data columns (total 28 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   HourUTC                     9984 non-null   object 
 1   HourDK                      9984 non-null   object 
 2   PriceArea                   9984 non-null   object 
 3   CentralPowerMWh             9984 non-null   float64
 4   LocalPowerMWh               9984 non-null   float64
 5   CommercialPowerMWh          9984 non-null   float64
 6   LocalPowerSelfConMWh        9984 non-null   float64
 7   OffshoreWindLt100MW_MWh     9984 non-null   float64
 8   OffshoreWindGe100MW_MWh     9984 non-null   float64
 9   OnshoreWindLt50kW_MWh       9984 non-null   float64
 10  OnshoreWindGe50kW_MWh       9984 non-null   float64
 11  HydroPowerMWh               9984 non-null   float64
 12  SolarPowerLt10kW_MWh        9984 non-null   float64
 13  SolarPowerGe10Lt40kW_MWh    9984 

In [7]:
prod["HourDK"] = pd.to_datetime(prod["HourDK"])

print("Rows:", len(prod))
print("Date range:", prod["HourDK"].min(), "→", prod["HourDK"].max())
print("Price areas:", prod["PriceArea"].unique())

Rows: 9984
Date range: 2024-01-01 00:00:00 → 2025-02-19 23:00:00
Price areas: ['DK2']


In [8]:
prod["offshore_wind_mwh"] = (
    prod["OffshoreWindLt100MW_MWh"] +
    prod["OffshoreWindGe100MW_MWh"]
)

prod["onshore_wind_mwh"] = (
    prod["OnshoreWindLt50kW_MWh"] +
    prod["OnshoreWindGe50kW_MWh"]
)

prod["solar_mwh"] = (
    prod["SolarPowerLt10kW_MWh"] +
    prod["SolarPowerGe10Lt40kW_MWh"] +
    prod["SolarPowerGe40kW_MWh"] +
    prod["SolarPowerSelfConMWh"]
)

prod["total_wind_mwh"] = (
    prod["offshore_wind_mwh"] +
    prod["onshore_wind_mwh"]
)

prod["renewable_generation_mwh"] = (
    prod["total_wind_mwh"] +
    prod["solar_mwh"]
)

prod["net_load_mwh"] = (
    prod["GrossConsumptionMWh"] -
    prod["renewable_generation_mwh"]
)

In [9]:
prod_clean = prod[
    [
        "HourDK",
        "HourUTC",
        "PriceArea",
        "GrossConsumptionMWh", #L
        "CentralPowerMWh", #G
        "LocalPowerMWh", #G
        "CommercialPowerMWh", #G
        "offshore_wind_mwh",
        "onshore_wind_mwh",
        "solar_mwh",
        "total_wind_mwh",
        "renewable_generation_mwh",
        "net_load_mwh",
        "ExchangeSE_MWh",
        "ExchangeGE_MWh",
        "ExchangeGreatBelt_MWh",
        "PowerToHeatMWh",
    ]
].copy()

prod_clean.head()

,HourDK,HourUTC,PriceArea,GrossConsumptionMWh,CentralPowerMWh,LocalPowerMWh,CommercialPowerMWh,offshore_wind_mwh,onshore_wind_mwh,solar_mwh,total_wind_mwh,renewable_generation_mwh,net_load_mwh,ExchangeSE_MWh,ExchangeGE_MWh,ExchangeGreatBelt_MWh,PowerToHeatMWh
0,2024-01-01 00:00:00,2023-12-31T23:00:00,DK2,1514.103305,299.433470,40.874795,199.417036,491.593168,195.941822,0.064280,687.534990,687.599270,826.504035,-1226.629258,966.118984,530.7,67.777938
1,2024-01-01 01:00:00,2024-01-01T00:00:00,DK2,1489.812135,294.905852,40.678208,200.459208,408.956069,178.526033,0.057712,587.482102,587.539814,902.272321,-1166.471250,986.252008,532.6,72.936872
2,2024-01-01 02:00:00,2024-01-01T01:00:00,DK2,1464.270618,293.800519,41.743095,198.599972,522.981768,185.398642,0.048805,708.380410,708.429215,755.841403,-1311.097968,986.575000,531.6,71.830910
3,2024-01-01 03:00:00,2024-01-01T02:00:00,DK2,1421.999512,275.398603,46.746541,201.408134,533.141367,199.392933,0.052524,732.534300,732.586824,689.412688,-1365.566234,986.662992,529.2,69.777540
4,2024-01-01 04:00:00,2024-01-01T03:00:00,DK2,1389.341870,275.347865,44.680464,202.559328,632.001366,214.987345,0.053213,846.988711,847.041924,542.299946,-1472.738500,986.767000,490.7,65.599544


In [10]:
prod_clean.isna().sum()

HourDK                      0
HourUTC                     0
PriceArea                   0
GrossConsumptionMWh         0
CentralPowerMWh             0
LocalPowerMWh               0
CommercialPowerMWh          0
offshore_wind_mwh           0
onshore_wind_mwh            0
solar_mwh                   0
total_wind_mwh              0
renewable_generation_mwh    0
net_load_mwh                0
ExchangeSE_MWh              0
ExchangeGE_MWh              0
ExchangeGreatBelt_MWh       0
PowerToHeatMWh              0
dtype: int64

In [11]:
prod_clean.to_csv(
    "../data/processed/dk2_production_consumption_clean.csv",
    index=False
)